# The potential-outcomes framework: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

The code from the module's web page, plus short exercises (cells marked `# YOUR CODE HERE`) and the regressions in `statsmodels`. Nothing to download: every population here is simulated, so we know both potential outcomes of every unit and can compute each term of the decompositions exactly.

**In Colab:** *File → Save a copy in Drive* first. Run cells with **Shift+Enter**.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

## Part A. The four units

`decompose` computes every term of both decompositions. With real data you could compute only the SDO: half of `Y1` and `Y0` would be missing.

In [ ]:
def decompose(D, Y1, Y0):
    """Every term of both decompositions; possible only because we know Y1 and Y0."""
    D, Y1, Y0 = map(np.asarray, (D, Y1, Y0))
    tau, pi = Y1 - Y0, D.mean()
    Y = D * Y1 + (1 - D) * Y0                     # the switching equation
    att, atu = tau[D == 1].mean(), tau[D == 0].mean()
    return dict(SDO=Y[D == 1].mean() - Y[D == 0].mean(), ATE=tau.mean(), ATT=att, ATU=atu,
                SB=Y0[D == 1].mean() - Y0[D == 0].mean(), HTE=(1 - pi) * (att - atu))

Y1, Y0 = [3, 1, 1, 1], [0, 1, 0, 1]
for name, D in [("A", [1, 1, 0, 0]), ("B", [0, 1, 0, 1]), ("C", [1, 0, 1, 0])]:
    print(name, {k: round(v, 2) for k, v in decompose(D, Y1, Y0).items()})

**Exercise A.** There are 14 ways to treat at least one and at most three of the four units. Loop over all of them (hint: `itertools.product([0, 1], repeat=4)`), check that both decompositions hold every time, and find the assignment with the largest SDO. Is it the one with the largest ATT?

In [ ]:
# YOUR CODE HERE

## Part B. 100,000 stores: self-selection vs. a coin flip

$Y_{0i}$ = weekly sales without a promotion; the effect $\tau_i$ is larger for stores with a bigger customer base.

In [ ]:
rng = np.random.default_rng(436)
N = 100_000
base = rng.normal(50, 10, N)                              # customer base
y0 = base + rng.normal(0, 8, N)                           # sales without the promotion
tau = 5 + 0.2 * (base - 50) + rng.normal(0, 3, N)         # bigger stores respond more
d_self = (base + rng.normal(0, 6, N) > 55).astype(int)    # big stores opt in
d_coin = rng.integers(0, 2, N)                            # random assignment
for name, d in [("self-selection", d_self), ("coin flip", d_coin)]:
    print(name, {k: round(v, 2) for k, v in decompose(d, y0 + tau, y0).items()})

**Exercise B.** Make the stores' choice noisier: replace the noise standard deviation 6 by each of 0, 3, 6, 12, 25, 50, 100, 1000 (keep the threshold at 55) and plot the selection bias and the heterogeneity term against the noise. What happens as the noise grows, and why?

In [ ]:
# YOUR CODE HERE

## Part C. One experiment, three regressions

A random sample of 2,000 stores, a coin flip for each, then: no controls; a pre-treatment covariate (a noisy measure of the customer base); a post-treatment variable (foot traffic, which the promotion raises).

In [ ]:
rng = np.random.default_rng(20)
n = 2_000
base = rng.normal(50, 10, n)
y0 = base + rng.normal(0, 8, n)
tau = 5 + 0.2 * (base - 50) + rng.normal(0, 3, n)
d = rng.integers(0, 2, n)                          # the coin flip
y = y0 + tau * d
df = pd.DataFrame({"y": y, "d": d,
                   "x_pre": base + rng.normal(0, 2, n),          # measured before the promotion
                   "m_post": 0.6 * y + rng.normal(0, 4, n)})     # foot traffic: moved by the promotion

print(f"true ATE in this sample: {tau.mean():.2f}")
print(f"difference in means:     {df.y[df.d == 1].mean() - df.y[df.d == 0].mean():.2f}")
for f in ["y ~ d", "y ~ d + x_pre", "y ~ d + m_post"]:
    m = smf.ols(f, data=df).fit()
    print(f"{f:16s} effect {m.params['d']:6.2f}   SE {m.bse['d']:.2f}")

**Exercise C.** Repeat the experiment 500 times (new stores and new coin flips each time) and collect the three estimates of the effect of `d`. Report the mean and standard deviation of each. Which is unbiased, which is the most precise, and which is biased?

In [ ]:
# YOUR CODE HERE